# Using Sentence Transformers and Hugging Face

## SentenceTransformers for paragraph similarities

`pip install sentence-transformers`

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
p1 = "Supply and install reinforced concrete slab 20MPa"
p2 = "Installation of 20 MPa RCC slab"

v1 = model.encode(p1)
v2 = model.encode(p2)

similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(similarity)

# Hugging Face Transformers with Torch

`pip install transformers torch`

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Load model (good balance of quality/speed)
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / \
           torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def embed(text):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    )

    with torch.no_grad():
        model_output = model(**encoded)

    sentence_embedding = mean_pooling(model_output, encoded["attention_mask"])

    # Normalize for cosine similarity
    sentence_embedding = F.normalize(sentence_embedding, p=2, dim=1)
    return sentence_embedding

In [ ]:
# Example paragraphs
p1 = "Supply and install reinforced concrete slab 20MPa including formwork."
p2 = "Installation of 20 MPa RCC slab with shuttering."

v1 = embed(p1)
v2 = embed(p2)

similarity = torch.matmul(v1, v2.T)
print("Similarity:", similarity.item())